In [2]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Strictly targeting Page 1
target_url = "https://www.mjla.gov.om/legislation/1/page/1"
base_url = "https://www.mjla.gov.om/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

output_folder = "MJLA_2025_Only_Page1"
os.makedirs(output_folder, exist_ok=True)

try:
    print(f"Connecting to: {target_url}...")
    response = requests.get(target_url, headers=headers, verify=False, timeout=20)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Website ke saare document blocks/thumbnails find karein
    documents = soup.find_all('div', class_='_df_thumb')
    
    print(f"Total {len(documents)} document blocks found on page. Filtering for 2025...")
    download_count = 0

    for index, doc in enumerate(documents, start=1):
        pdf_url = doc.get('data-source')
        if not pdf_url:
            continue
            
        # Pura card element (article ya parent container) check karne ke liye taaki date text mil sake
        parent_card = doc.find_parent('article') or doc.find_parent('div', class_='post-content') or doc.find_parent()
        parent_text = parent_card.get_text() if parent_card else ""
        
        # CONDITION: Sirf tab download karein jab parent card me ya PDF link me '2025' maujood ho
        if "2025" in parent_text or "2025" in pdf_url:
            full_pdf_url = urljoin(base_url, pdf_url)
            
            # Windows OS safe filename extraction
            raw_filename = pdf_url.split('/')[-1].split('?')[0]
            if not raw_filename.endswith('.pdf'):
                raw_filename += ".pdf"
                
            clean_filename = f"2025_doc_{index}_{raw_filename}"
            file_path = os.path.join(output_folder, clean_filename)
            
            print(f"-> Match Found! Downloading 2025 document: {clean_filename}")
            try:
                pdf_download = requests.get(full_pdf_url, headers=headers, verify=False, timeout=20)
                if pdf_download.status_code == 200:
                    with open(file_path, 'wb') as f:
                        f.write(pdf_download.content)
                    download_count += 1
                else:
                    print(f"    Failed to download. Status code: {pdf_download.status_code}")
            except Exception as file_err:
                print(f"    Error saving file: {file_err}")
        else:
            # 2026 ya baaki dusre years ke documents ko skip kar dega
            continue

    print(f"\nTask Finished! Total {download_count} files of year 2025 saved inside '{output_folder}' folder.")

except Exception as e:
    print(f"An error occurred: {e}")

Connecting to: https://www.mjla.gov.om/legislation/1/page/1...


c:\Users\farha\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mjla.gov.om'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Total 24 document blocks found on page. Filtering for 2025...
-> Match Found! Downloading 2025 document: 2025_doc_23_Book74147.pdf


c:\Users\farha\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mjla.gov.om'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


-> Match Found! Downloading 2025 document: 2025_doc_24_Book533070.pdf


c:\Users\farha\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mjla.gov.om'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(



Task Finished! Total 2 files of year 2025 saved inside 'MJLA_2025_Only_Page1' folder.
